# 07·04 — Dataset Creation for Fine-Tuning

> **Module:** 07 – Fine-Tuning  
> **Notebook:** 4 of 4  
> **Objective:** Build production-quality fine-tuning datasets from scratch — sourcing, formatting, filtering, deduplicating, and splitting with rigour.

---

## 🎯 Learning Objectives

1. Understand the data flywheel: why data quality dominates quantity for SFT
2. Implement three dataset creation pipelines: human-curated, synthetic (Self-Instruct), and distillation
3. Build a robust quality filtering pipeline (length, diversity, perplexity, deduplication)
4. Format and validate datasets for HuggingFace compatibility
5. Perform proper train/validation/test splits with contamination checks

---

## 1. The Data Landscape for Instruction Tuning

```
DATASET SOURCES (ordered by quality, most expensive → cheapest)
────────────────────────────────────────────────────────────────

1. Human expert annotations        ████████████  Highest quality, expensive
   e.g., InstructGPT labellers     Cost: $50–200/hour per labeller
   Volume: 10k–100k examples       Time: weeks–months

2. Filtered human data             ██████████    High quality if filtered well
   e.g., Stack Overflow, Reddit    Cost: near-zero (public data)
   Volume: 100k–10M                Time: days (engineering)

3. LLM distillation                ████████      Scalable, good if model is good
   e.g., Alpaca, Dolly, Orca       Cost: API costs
   Volume: 10k–1M                  Time: hours–days

4. Synthetic via Self-Instruct     ██████        Diverse, needs quality filter
   e.g., generated from seed tasks Cost: moderate API costs
   Volume: unlimited               Time: hours

5. Untreated web scrape            ████          High noise, needs heavy filtering
   e.g., raw CommonCrawl           Cost: near-zero
   Volume: billions                Time: weeks of engineering

KEY INSIGHT (Alpagasus, 2023): 9k high-quality examples > 52k noisy Alpaca examples
```

---

## 2. Dataset Anatomy: What Makes a Good Example?

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install datasets transformers sentence-transformers faiss-cpu matplotlib seaborn pandas tqdm -q

import json
import re
import hashlib
import random
import math
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Optional, Tuple, Set
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm

import torch
from datasets import Dataset, DatasetDict
from sentence_transformers import SentenceTransformer

random.seed(42)
np.random.seed(42)
print("Setup complete.")

In [ ]:
@dataclass
class Instruction:
    """
    Single instruction-tuning example. The fundamental unit of your dataset.
    All fields beyond instruction/response are metadata for filtering and analysis.
    """
    instruction: str
    response:    str
    input:       str = ""           # Optional context for the instruction
    source:      str = "unknown"    # Provenance: "human", "gpt4", "synthetic", etc.
    category:    str = "general"    # Task category for stratified splits
    language:    str = "en"
    metadata:    Dict = field(default_factory=dict)

    def to_dict(self) -> Dict:
        return {
            "instruction": self.instruction,
            "input":       self.input,
            "response":    self.response,
            "source":      self.source,
            "category":    self.category,
        }

    def format_alpaca(self) -> str:
        if self.input:
            return (
                f"### Instruction:\n{self.instruction}\n\n"
                f"### Input:\n{self.input}\n\n"
                f"### Response:\n{self.response}"
            )
        return f"### Instruction:\n{self.instruction}\n\n### Response:\n{self.response}"

    def format_chatml(self) -> str:
        user_content = self.instruction
        if self.input:
            user_content += f"\n\nContext:\n{self.input}"
        return (
            f"<|im_start|>user\n{user_content}<|im_end|>\n"
            f"<|im_start|>assistant\n{self.response}<|im_end|>"
        )

    @property
    def total_length(self) -> int:
        return len(self.instruction) + len(self.input) + len(self.response)

    @property
    def response_length(self) -> int:
        return len(self.response.split())


# Annotated examples of good vs poor quality
good_example = Instruction(
    instruction="Explain the bias-variance trade-off in machine learning.",
    response=(
        "The bias-variance trade-off describes two sources of model error:\n\n"
        "**Bias** is error from overly simplistic assumptions — the model is "
        "too constrained to capture the true pattern (underfitting). High bias "
        "means consistent but wrong predictions.\n\n"
        "**Variance** is error from sensitivity to small fluctuations in training data "
        "— the model learns noise instead of signal (overfitting). High variance means "
        "inconsistent predictions across different training sets.\n\n"
        "The trade-off: increasing model complexity reduces bias but increases variance. "
        "The optimal model minimises total error = Bias² + Variance + Irreducible noise."
    ),
    source="human",
    category="explanation"
)

poor_example = Instruction(
    instruction="ML explain",
    response="ML is good. You should learn it.",
    source="synthetic",
    category="general"
)

print("Good example:")
print(f"  Instruction words: {len(good_example.instruction.split())}")
print(f"  Response words:    {good_example.response_length}")
print(f"  Preview: {good_example.response[:100]}...")
print()
print("Poor example:")
print(f"  Instruction words: {len(poor_example.instruction.split())}")
print(f"  Response words:    {poor_example.response_length}")
print(f"  Preview: {poor_example.response}")

## 3. Pipeline 1: Synthetic Generation via Self-Instruct

Self-Instruct (Wang et al., 2022) bootstraps a large dataset from a small seed set by using an LLM to generate new instructions and responses. It's how Alpaca was created.

In [ ]:
class SelfInstructPipeline:
    """
    Implements the Self-Instruct data generation pipeline.

    Original paper method:
    1. Start with ~175 human-written seed tasks
    2. Sample 8 tasks from the pool (some seed, some generated)
    3. Prompt the LLM to generate new instructions inspired by the sample
    4. Classify as classification vs open-ended
    5. Generate inputs and outputs for each new instruction
    6. Filter low-quality outputs
    7. Add surviving examples to the pool, repeat
    """

    # Seed tasks covering diverse capability areas
    SEED_TASKS = [
        # Explanation tasks
        {"instruction": "Explain {concept} in simple terms.",
         "category": "explanation",
         "slots": ["gradient descent", "attention mechanisms", "overfitting",
                   "regularisation", "backpropagation", "batch normalisation"]},
        # Comparison tasks
        {"instruction": "Compare and contrast {A} and {B}.",
         "category": "comparison",
         "slots": [("LSTM", "Transformer"), ("ReLU", "GELU"),
                   ("Adam", "SGD"), ("BERT", "GPT")]},
        # Code generation tasks
        {"instruction": "Write a Python function that {task}.",
         "category": "code",
         "slots": ["computes softmax", "implements a simple neural network layer",
                   "calculates cosine similarity between two vectors",
                   "implements gradient descent from scratch"]},
        # Step-by-step tasks
        {"instruction": "Walk me through how {process} works step by step.",
         "category": "procedural",
         "slots": ["transformer self-attention works",
                   "a model is trained", "backpropagation computes gradients",
                   "beam search generates text"]},
        # Troubleshooting
        {"instruction": "My model is {symptom}. What could be wrong and how do I fix it?",
         "category": "debugging",
         "slots": ["overfitting badly", "not converging", "producing NaN losses",
                   "very slow to train"]},
    ]

    # Template responses (used for demo; replace with actual LLM calls in production)
    RESPONSE_TEMPLATES = {
        "explanation": (
            "{concept} refers to the process by which neural networks learn from data. "
            "At its core, it involves computing gradients and updating parameters iteratively. "
            "The key insight is that small, incremental updates guided by the data's signal "
            "allow the model to improve its predictions over many training steps."
        ),
        "comparison": (
            "Both approaches have their strengths. The first excels at sequential processing "
            "and handles temporal dependencies well. The second is more parallelisable "
            "and captures long-range dependencies more effectively through attention. "
            "For modern NLP tasks, the second is generally preferred due to scalability."
        ),
        "code": (
            "```python\n"
            "import numpy as np\n\n"
            "def compute_operation(x):\n"
            "    \"\"\"Implements the specified mathematical operation.\"\"\"\n"
            "    # Numerical stability: subtract max before exponentiation\n"
            "    x_shifted = x - np.max(x, axis=-1, keepdims=True)\n"
            "    exp_x = np.exp(x_shifted)\n"
            "    return exp_x / exp_x.sum(axis=-1, keepdims=True)\n"
            "```"
        ),
        "procedural": (
            "Step 1: Initialise the process by preparing the inputs.\n"
            "Step 2: Compute the forward pass, applying transformations to produce output.\n"
            "Step 3: Calculate the loss by comparing output to the target.\n"
            "Step 4: Propagate gradients backward through the computational graph.\n"
            "Step 5: Update parameters using the computed gradients."
        ),
        "debugging": (
            "This issue typically stems from one of three root causes:\n\n"
            "1. **Learning rate** — Try reducing it by 10x as a first step.\n"
            "2. **Data quality** — Check for outliers, NaN values, and class imbalance.\n"
            "3. **Architecture** — Verify layer sizes and activation functions are appropriate.\n\n"
            "Start by adding diagnostic logging to pinpoint exactly where things go wrong."
        ),
    }

    def generate_from_seeds(
        self,
        n_examples: int = 50,
        seed: int = 42
    ) -> List[Instruction]:
        """
        Generate synthetic instructions from seed templates.
        In production, replace response generation with LLM API calls.
        """
        rng = random.Random(seed)
        examples = []

        for _ in range(n_examples):
            template = rng.choice(self.SEED_TASKS)
            slot_value = rng.choice(template['slots'])

            # Fill the instruction template
            if isinstance(slot_value, tuple):
                instruction = template['instruction'].replace('{A}', slot_value[0]).replace('{B}', slot_value[1])
                slot_str = f"{slot_value[0]} vs {slot_value[1]}"
            else:
                instruction = template['instruction'].replace(
                    '{concept}', slot_value
                ).replace(
                    '{task}', slot_value
                ).replace(
                    '{process}', slot_value
                ).replace(
                    '{symptom}', slot_value
                )
                slot_str = slot_value

            response_tmpl = self.RESPONSE_TEMPLATES[template['category']]
            response = response_tmpl.replace('{concept}', slot_str)

            # Add small variations to avoid exact duplicates
            suffixes = ["", " Please be concise.", " Use an example.",
                        " Assume I'm a beginner.", " Include key trade-offs."]
            instruction += rng.choice(suffixes)

            examples.append(Instruction(
                instruction=instruction.strip(),
                response=response,
                source="self_instruct",
                category=template['category'],
            ))

        return examples


pipeline = SelfInstructPipeline()
synthetic_data = pipeline.generate_from_seeds(n_examples=200)

print(f"Generated {len(synthetic_data)} synthetic examples")
print()

# Category distribution
cat_counts = Counter(e.category for e in synthetic_data)
print("Category distribution:")
for cat, count in cat_counts.most_common():
    bar = '█' * (count // 2)
    print(f"  {cat:<15} {count:>3}  {bar}")

print()
print("Sample:")
for ex in synthetic_data[:2]:
    print(f"  [{ex.category}] {ex.instruction}")
    print(f"   → {ex.response[:80]}...")
    print()

## 4. Pipeline 2: Distillation from a Stronger Model

Use a capable teacher model (e.g., GPT-4, Claude) to generate responses for a set of carefully chosen prompts. This is how Alpaca, Dolly, and Orca were created.

In [ ]:
class DistillationPipeline:
    """
    Knowledge distillation: use a strong teacher model to label a dataset.

    The Orca paper (2023) showed that asking the teacher to show its
    step-by-step reasoning (system prompts like 'Think step by step') produces
    dramatically better student models than direct answer distillation.
    """

    SYSTEM_PROMPTS = {
        "direct": "You are a helpful assistant. Answer clearly and concisely.",

        "chain_of_thought": (
            "You are a helpful assistant. Before giving your final answer, "
            "think step by step through the problem. Show your reasoning explicitly."
        ),

        "expert": (
            "You are an expert ML engineer with 10 years of experience. "
            "Give thorough, technically accurate answers with concrete examples. "
            "Explain trade-offs and mention common pitfalls."
        ),

        "socratic": (
            "You are a thoughtful teacher. Explain concepts by building from "
            "first principles. Use analogies to make abstract ideas concrete. "
            "Anticipate and address likely misconceptions."
        ),
    }

    def generate_distillation_prompt(
        self,
        instruction: str,
        system_style: str = "chain_of_thought"
    ) -> str:
        """Builds the API prompt for teacher model distillation."""
        system = self.SYSTEM_PROMPTS[system_style]
        return f"{system}\n\nUser: {instruction}\n\nAssistant:"

    def generate_response_with_api(
        self,
        instruction: str,
        api_client,  # anthropic.Anthropic() or openai.OpenAI()
        model: str = "claude-3-haiku-20240307",
        system_style: str = "chain_of_thought",
    ) -> str:
        """Call the teacher model API. Use this in production."""
        system = self.SYSTEM_PROMPTS[system_style]
        # For Anthropic:
        response = api_client.messages.create(
            model=model,
            max_tokens=1024,
            system=system,
            messages=[{"role": "user", "content": instruction}]
        )
        return response.content[0].text


# Demonstrate the impact of system prompts on response quality
distill = DistillationPipeline()

instruction = "Why does my neural network produce NaN loss?"

print(f"Instruction: {instruction}")
print()
for style, system_prompt in distill.SYSTEM_PROMPTS.items():
    full_prompt = distill.generate_distillation_prompt(instruction, style)
    print(f"System style: [{style}]")
    print(f"  System: {system_prompt[:80]}...")
    print(f"  Prompt length: {len(full_prompt)} chars")
    print()

print("─" * 60)
print("ORCA PAPER INSIGHT:")
print("Training on chain-of-thought responses from GPT-4 produced")
print("a 13B model that outperformed much larger models on reasoning.")
print("The 'show your work' system prompt was the key innovation.")

## 5. Quality Filtering Pipeline

Raw data — whether synthetic or scraped — contains noise, duplicates, and low-quality examples that hurt training. A filtering pipeline is mandatory.

In [ ]:
@dataclass
class FilterConfig:
    """Configuration for the quality filtering pipeline."""
    # Length filters
    min_instruction_words: int = 3
    max_instruction_words: int = 200
    min_response_words:    int = 10
    max_response_words:    int = 800

    # Content filters
    min_response_instruction_ratio: float = 1.5  # response must be longer than instruction
    max_response_instruction_ratio: float = 50.0  # guard against tiny instructions

    # Deduplication
    dedup_similarity_threshold: float = 0.85  # cosine similarity for near-dedup

    # Safety
    blocked_substrings: List[str] = field(default_factory=lambda: [
        "lorem ipsum", "[PLACEHOLDER]", "TODO:", "FIXME",
        "as an AI language model, I",
        "I cannot and will not",
        "I'm just an AI",
    ])


class QualityFilter:
    """
    Multi-stage quality filter for instruction-tuning datasets.

    Stages (in order — fail fast, skip expensive checks early):
    1. Length check          (fast, O(1))
    2. Ratio check           (fast, O(1))
    3. Content/safety check  (fast, O(n) in text length)
    4. Language detection    (medium)
    5. Near-deduplication    (slow — only if embedding model available)
    """

    def __init__(self, config: FilterConfig = None):
        self.config = config or FilterConfig()
        self.filter_counts = defaultdict(int)

    def check_length(self, ex: Instruction) -> Tuple[bool, str]:
        instr_words = len(ex.instruction.split())
        resp_words  = len(ex.response.split())
        c = self.config

        if instr_words < c.min_instruction_words:
            return False, f"instruction too short ({instr_words} words)"
        if instr_words > c.max_instruction_words:
            return False, f"instruction too long ({instr_words} words)"
        if resp_words < c.min_response_words:
            return False, f"response too short ({resp_words} words)"
        if resp_words > c.max_response_words:
            return False, f"response too long ({resp_words} words)"
        return True, "ok"

    def check_ratio(self, ex: Instruction) -> Tuple[bool, str]:
        instr_words = max(len(ex.instruction.split()), 1)
        resp_words  = len(ex.response.split())
        ratio = resp_words / instr_words

        if ratio < self.config.min_response_instruction_ratio:
            return False, f"response/instruction ratio too low ({ratio:.1f})"
        if ratio > self.config.max_response_instruction_ratio:
            return False, f"response/instruction ratio too high ({ratio:.1f})"
        return True, "ok"

    def check_content(self, ex: Instruction) -> Tuple[bool, str]:
        combined = (ex.instruction + " " + ex.response).lower()

        # Blocked substrings
        for blocked in self.config.blocked_substrings:
            if blocked.lower() in combined:
                return False, f"contains blocked substring: '{blocked}'"

        # Check for mostly non-alphabetic content
        alpha_ratio = sum(c.isalpha() for c in ex.response) / max(len(ex.response), 1)
        if alpha_ratio < 0.5:
            return False, f"response has low alphabetic ratio ({alpha_ratio:.2f})"

        # Check for excessive repetition
        words = ex.response.lower().split()
        if len(words) > 5:
            unique_ratio = len(set(words)) / len(words)
            if unique_ratio < 0.3:
                return False, f"response has high repetition ({unique_ratio:.2f} unique ratio)"

        return True, "ok"

    def check_instruction_ends_with_question_or_verb(self, ex: Instruction) -> Tuple[bool, str]:
        """Heuristic: good instructions tend to end with ? or start with action verbs."""
        instr = ex.instruction.strip()
        action_verbs = ['explain', 'describe', 'list', 'compare', 'write', 'create',
                        'implement', 'calculate', 'what', 'how', 'why', 'when', 'where',
                        'define', 'summarise', 'analyze', 'translate', 'give', 'show']
        first_word = instr.lower().split()[0] if instr else ""

        if instr.endswith('?') or first_word in action_verbs:
            return True, "ok"
        return True, "weak_instruction"  # warn but don't filter — too aggressive

    def filter(
        self,
        examples: List[Instruction],
        verbose: bool = True,
    ) -> Tuple[List[Instruction], Dict]:
        """Apply all filters. Returns (passing_examples, filter_stats)."""
        passing = []
        reasons = defaultdict(int)

        for ex in examples:
            checks = [
                self.check_length,
                self.check_ratio,
                self.check_content,
            ]
            passed = True
            for check in checks:
                ok, reason = check(ex)
                if not ok:
                    reasons[reason.split('(')[0].strip()] += 1  # normalise reason
                    passed = False
                    break

            if passed:
                passing.append(ex)

        stats = {
            'input':     len(examples),
            'passing':   len(passing),
            'filtered':  len(examples) - len(passing),
            'pass_rate': len(passing) / max(len(examples), 1),
            'reasons':   dict(reasons),
        }
        if verbose:
            print(f"Filter results: {stats['input']} → {stats['passing']} examples "
                  f"({stats['pass_rate']:.1%} pass rate)")
            print("  Filtered for:")
            for reason, count in sorted(reasons.items(), key=lambda x: -x[1]):
                print(f"    {reason:<45} {count:>4}")

        return passing, stats


# Add some deliberately bad examples to test filtering
bad_examples = [
    Instruction("AI", "OK.", source="noise"),   # too short
    Instruction("What?" * 50, "Fine." * 200, source="noise"),   # too long
    Instruction("What is ML?", "ML. ML. ML. ML. ML. ML. ML. ML. ML. ML.", source="noise"),  # repetitive
    Instruction("Help", "As an AI language model, I cannot provide that.", source="noise"),
    Instruction("Write code.", "1234567890" * 20, source="noise"),  # low alpha ratio
    Instruction("Great question!", "Great question!" * 5, source="noise"),  # copied instruction
]

all_data = synthetic_data + bad_examples
filter_ = QualityFilter(FilterConfig(
    min_response_words=8,  # relaxed for demo
    min_instruction_words=2,
))
filtered_data, filter_stats = filter_.filter(all_data)

## 6. Deduplication

Duplicates and near-duplicates cause models to memorise specific examples rather than generalise. Both exact and semantic deduplication are necessary.

In [ ]:
class Deduplicator:
    """
    Two-stage deduplication:
    1. Exact dedup: hash-based, O(n) — catches copy-paste duplicates
    2. Semantic dedup: embedding cosine similarity, O(n²) or approximate
       — catches paraphrastic duplicates
    """

    def __init__(self, embed_model_name: str = "all-MiniLM-L6-v2"):
        self._embedder = None
        self._embed_model_name = embed_model_name

    @property
    def embedder(self):
        if self._embedder is None:
            print(f"Loading embedding model '{self._embed_model_name}'...")
            self._embedder = SentenceTransformer(self._embed_model_name)
        return self._embedder

    def exact_dedup(
        self,
        examples: List[Instruction],
        field: str = "instruction",  # dedup on instruction text
    ) -> Tuple[List[Instruction], int]:
        """
        Hash-based exact deduplication.
        Normalises whitespace and lowercases before hashing.
        """
        seen_hashes: Set[str] = set()
        unique = []
        n_dupes = 0

        for ex in examples:
            text = getattr(ex, field)
            # Normalise: lowercase, collapse whitespace
            normalised = " ".join(text.lower().split())
            h = hashlib.md5(normalised.encode()).hexdigest()

            if h not in seen_hashes:
                seen_hashes.add(h)
                unique.append(ex)
            else:
                n_dupes += 1

        return unique, n_dupes

    def semantic_dedup(
        self,
        examples: List[Instruction],
        threshold: float = 0.92,
        batch_size: int = 64,
    ) -> Tuple[List[Instruction], int]:
        """
        Embedding-based near-deduplication.

        Removes examples whose instruction has cosine similarity > threshold
        with any already-accepted example. Greedy: processes in order.

        O(n²) — for large datasets use FAISS approximate nearest neighbours.
        """
        if not examples:
            return examples, 0

        instructions = [ex.instruction for ex in examples]
        print(f"  Computing embeddings for {len(instructions)} examples...")

        embeddings = self.embedder.encode(
            instructions,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

        # Greedy dedup: keep if no existing example has sim > threshold
        kept_indices = [0]
        kept_embs = embeddings[[0]]

        for i in range(1, len(examples)):
            emb = embeddings[i]
            max_sim = float((kept_embs @ emb).max())
            if max_sim < threshold:
                kept_indices.append(i)
                kept_embs = np.vstack([kept_embs, emb[None]])

        unique = [examples[i] for i in kept_indices]
        n_removed = len(examples) - len(unique)
        return unique, n_removed


dedup = Deduplicator()

# Add some intentional duplicates
duplicated = filtered_data + filtered_data[:20]

print(f"Before dedup: {len(duplicated)} examples")
after_exact, n_exact = dedup.exact_dedup(duplicated)
print(f"After exact dedup: {len(after_exact)} examples (removed {n_exact})")

# Semantic dedup on a smaller sample for demo
sample_for_dedup = after_exact[:60]
after_semantic, n_semantic = dedup.semantic_dedup(sample_for_dedup, threshold=0.90)
print(f"After semantic dedup (sample): {len(after_semantic)} examples (removed {n_semantic})")

## 7. Dataset Analysis & Visualisation

In [ ]:
def analyse_dataset(examples: List[Instruction], title: str = "Dataset") -> Dict:
    """Comprehensive dataset statistics and visualisation."""
    if not examples:
        print("Empty dataset")
        return {}

    instr_lens = [len(e.instruction.split()) for e in examples]
    resp_lens  = [len(e.response.split())    for e in examples]
    ratios     = [r/max(i,1) for i, r in zip(instr_lens, resp_lens)]
    categories = Counter(e.category for e in examples)
    sources    = Counter(e.source   for e in examples)

    stats = {
        'n_examples':      len(examples),
        'instr_len_mean':  np.mean(instr_lens),
        'instr_len_p50':   np.percentile(instr_lens, 50),
        'instr_len_p95':   np.percentile(instr_lens, 95),
        'resp_len_mean':   np.mean(resp_lens),
        'resp_len_p50':    np.percentile(resp_lens, 50),
        'resp_len_p95':    np.percentile(resp_lens, 95),
        'ratio_mean':      np.mean(ratios),
        'categories':      dict(categories),
        'sources':         dict(sources),
    }

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    # Instruction length distribution
    ax = axes[0, 0]
    ax.hist(instr_lens, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(instr_lens), color='red', linestyle='--',
               label=f'Mean={np.mean(instr_lens):.0f}')
    ax.set_xlabel('Instruction length (words)')
    ax.set_ylabel('Count')
    ax.set_title('Instruction Length Distribution')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    # Response length distribution
    ax = axes[0, 1]
    ax.hist(resp_lens, bins=30, color='darkorange', alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(resp_lens), color='red', linestyle='--',
               label=f'Mean={np.mean(resp_lens):.0f}')
    ax.set_xlabel('Response length (words)')
    ax.set_ylabel('Count')
    ax.set_title('Response Length Distribution')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    # Instruction vs Response scatter
    ax = axes[0, 2]
    scatter_colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))
    cat_to_color   = dict(zip(categories.keys(), scatter_colors))
    for ex, il, rl in zip(examples, instr_lens, resp_lens):
        ax.scatter(il, rl, color=cat_to_color[ex.category], alpha=0.4, s=20)
    ax.set_xlabel('Instruction length (words)')
    ax.set_ylabel('Response length (words)')
    ax.set_title('Instruction vs Response Length')
    legend_patches = [mpatches.Patch(color=cat_to_color[c], label=c) for c in categories]
    ax.legend(handles=legend_patches, fontsize=7, ncol=2)
    ax.grid(alpha=0.3)

    # Category pie
    ax = axes[1, 0]
    cats, counts = zip(*categories.most_common())
    ax.pie(counts, labels=cats, autopct='%1.1f%%',
           colors=plt.cm.Set2(np.linspace(0, 1, len(cats))))
    ax.set_title('Category Distribution')

    # Source bar chart
    ax = axes[1, 1]
    src_names, src_counts = zip(*sources.most_common()) if sources else (["none"], [0])
    ax.bar(src_names, src_counts, color='slateblue', alpha=0.85)
    ax.set_ylabel('Count')
    ax.set_title('Data Source Distribution')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(alpha=0.3, axis='y')

    # Response/instruction ratio
    ax = axes[1, 2]
    ax.hist(ratios, bins=30, color='#2ca02c', alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(ratios), color='red', linestyle='--',
               label=f'Mean ratio={np.mean(ratios):.1f}x')
    ax.axvline(1.0, color='orange', linestyle=':', alpha=0.7, label='ratio=1 (same length)')
    ax.set_xlabel('Response / Instruction length ratio')
    ax.set_ylabel('Count')
    ax.set_title('Response-to-Instruction Ratio')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    plt.suptitle(f'{title} — {len(examples):,} examples', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/07_dataset_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nSummary statistics for '{title}':")
    print(f"  Examples:           {stats['n_examples']:,}")
    print(f"  Instruction length: mean={stats['instr_len_mean']:.0f} | p50={stats['instr_len_p50']:.0f} | p95={stats['instr_len_p95']:.0f} words")
    print(f"  Response length:    mean={stats['resp_len_mean']:.0f} | p50={stats['resp_len_p50']:.0f} | p95={stats['resp_len_p95']:.0f} words")
    print(f"  Resp/instr ratio:   mean={stats['ratio_mean']:.1f}x")

    return stats


stats = analyse_dataset(after_exact, title="Filtered & Deduplicated Dataset")

## 8. Train / Validation / Test Splits

In [ ]:
def create_splits(
    examples: List[Instruction],
    train_ratio:  float = 0.80,
    val_ratio:    float = 0.10,
    test_ratio:   float = 0.10,
    stratify_by:  Optional[str] = "category",
    seed:         int = 42,
) -> DatasetDict:
    """
    Create stratified train/val/test splits and package as HuggingFace DatasetDict.

    Stratified splitting ensures each split has proportional representation
    of every category — critical to avoid biased evaluation.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Ratios must sum to 1.0"

    rng = random.Random(seed)

    if stratify_by:
        # Group by category
        groups = defaultdict(list)
        for ex in examples:
            groups[getattr(ex, stratify_by)].append(ex)

        train_exs, val_exs, test_exs = [], [], []
        for group_name, group_exs in groups.items():
            rng.shuffle(group_exs)
            n = len(group_exs)
            n_train = int(n * train_ratio)
            n_val   = int(n * val_ratio)
            train_exs.extend(group_exs[:n_train])
            val_exs.extend(group_exs[n_train:n_train + n_val])
            test_exs.extend(group_exs[n_train + n_val:])
    else:
        rng.shuffle(examples)
        n = len(examples)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        train_exs = examples[:n_train]
        val_exs   = examples[n_train:n_train + n_val]
        test_exs  = examples[n_train + n_val:]

    # Shuffle each split
    for split in [train_exs, val_exs, test_exs]:
        rng.shuffle(split)

    def to_hf_dataset(exs: List[Instruction], fmt: str = "alpaca") -> Dataset:
        records = []
        for ex in exs:
            records.append({
                **ex.to_dict(),
                "text": ex.format_alpaca(),
                "text_chatml": ex.format_chatml(),
            })
        return Dataset.from_list(records)

    dataset_dict = DatasetDict({
        "train":      to_hf_dataset(train_exs),
        "validation": to_hf_dataset(val_exs),
        "test":       to_hf_dataset(test_exs),
    })

    return dataset_dict, train_exs, val_exs, test_exs


dataset_dict, train_exs, val_exs, test_exs = create_splits(
    after_exact,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    stratify_by="category",
)

print("Split sizes:")
for split_name, split_ds in dataset_dict.items():
    print(f"  {split_name:<12} {len(split_ds):>4} examples")

# Verify stratification quality
print("\nCategory distribution per split (should be roughly equal %):")
print(f"{'Category':<15} {'Train%':>8} {'Val%':>8} {'Test%':>8}")
print("-" * 42)

all_cats = set(e.category for e in after_exact)
for cat in sorted(all_cats):
    t_pct = 100 * sum(1 for e in train_exs if e.category == cat) / max(len(train_exs), 1)
    v_pct = 100 * sum(1 for e in val_exs   if e.category == cat) / max(len(val_exs),   1)
    s_pct = 100 * sum(1 for e in test_exs  if e.category == cat) / max(len(test_exs),  1)
    print(f"  {cat:<15} {t_pct:>7.1f}% {v_pct:>7.1f}% {s_pct:>7.1f}%")

In [ ]:
# ── Contamination check: ensure test set instructions didn't leak into train ──

def check_contamination(
    train: List[Instruction],
    test:  List[Instruction],
    embed_model=None,
    similarity_threshold: float = 0.85,
) -> Dict:
    """
    Check whether test examples are too similar to training examples.

    Contamination = test examples the model may have 'memorised' during training,
    leading to inflated eval metrics.

    For final benchmarks, also check for substring overlap (simpler, faster).
    """
    # Fast exact-string check
    train_instrs = {" ".join(e.instruction.lower().split()) for e in train}
    exact_leaks = [
        e for e in test
        if " ".join(e.instruction.lower().split()) in train_instrs
    ]

    # N-gram overlap check (finds partial contamination)
    def ngrams(text: str, n: int = 8) -> Set:
        words = text.lower().split()
        return {tuple(words[i:i+n]) for i in range(len(words) - n + 1)}

    train_ngrams = set()
    for e in train:
        train_ngrams.update(ngrams(e.instruction))

    near_leaks = [
        e for e in test
        if ngrams(e.instruction) & train_ngrams  # any shared 8-gram
    ]

    return {
        'exact_leaks': len(exact_leaks),
        'near_leaks_8gram': len(near_leaks),
        'contamination_rate': len(near_leaks) / max(len(test), 1),
        'examples': near_leaks[:3],
    }


contamination = check_contamination(train_exs, test_exs)
print("Contamination check results:")
print(f"  Exact duplicates (train→test): {contamination['exact_leaks']}")
print(f"  Near-duplicates (8-gram):      {contamination['near_leaks_8gram']}")
print(f"  Contamination rate:            {contamination['contamination_rate']:.1%}")
if contamination['near_leaks_8gram'] == 0:
    print("  ✅ Clean split — no contamination detected")
else:
    print("  ⚠️  Contaminated! Remove overlapping examples from test set.")

In [ ]:
# ── Save the final dataset ────────────────────────────────────────────────

output_dir = Path("./final_dataset")
output_dir.mkdir(exist_ok=True)

# Save as JSON (human-readable, easy to inspect and edit)
for split_name in ['train', 'validation', 'test']:
    split_exs = {'train': train_exs, 'validation': val_exs, 'test': test_exs}[split_name]
    out_path = output_dir / f"{split_name}.jsonl"
    with open(out_path, 'w') as f:
        for ex in split_exs:
            f.write(json.dumps(ex.to_dict()) + '\n')
    print(f"Saved {split_name}: {len(split_exs)} examples → {out_path}")

# Save as HuggingFace dataset (faster loading, built-in batching)
dataset_dict.save_to_disk(str(output_dir / "hf_dataset"))
print(f"\nSaved HuggingFace dataset → {output_dir / 'hf_dataset'}")

# Show final dataset card info
print("\n" + "=" * 55)
print("DATASET CARD")
print("=" * 55)
print(f"Name:        my_instruct_dataset")
print(f"Total:       {len(after_exact)} examples")
print(f"Train:       {len(train_exs)} examples")
print(f"Validation:  {len(val_exs)} examples")
print(f"Test:        {len(test_exs)} examples")
print(f"Sources:     {set(e.source for e in after_exact)}")
print(f"Categories:  {set(e.category for e in after_exact)}")
print(f"Format:      Alpaca + ChatML")
print(f"Contaminated: {'No' if contamination['exact_leaks'] == 0 else 'Yes — fix before publishing!'}")

## 9. The Whole Pipeline: Production Summary

```
PRODUCTION DATASET CREATION PIPELINE
═══════════════════════════════════════════════════════════

  [Raw sources]
      │
      ├── Human annotations    (curated, expensive, high quality)
      ├── Filtered web data    (StackOverflow, Reddit, docs)
      ├── LLM distillation     (GPT-4/Claude teacher model)
      └── Self-Instruct        (synthetic, needs heavy filtering)
      │
      ▼
  [Stage 1: Rule-based filtering]           ← Remove obvious garbage fast
      Length check  |  Ratio check  |  Content check  |  Safety check
      │
      ▼
  [Stage 2: Exact deduplication]            ← Hash-based, O(n)
      │
      ▼
  [Stage 3: Semantic deduplication]         ← Embedding cosine sim, O(n²)
      │
      ▼
  [Stage 4: Quality scoring]                ← Optional: LLM-based quality score
      Alpagasus method: GPT-4 rates 1-5, keep top N
      │
      ▼
  [Stage 5: Dataset analysis]               ← Check distributions, find gaps
      │
      ▼
  [Stage 6: Stratified splits]              ← Train/val/test with contamination check
      │
      ▼
  [Final dataset]  →  JSONL + HuggingFace format  →  Train!
```

## 10. Exercises

1. **Alpagasus Replication**: Take 200 Alpaca examples. Use an LLM to rate each from 1–5 on accuracy and usefulness. Keep only examples rated ≥4. Train on this filtered set vs the full set. Which produces a better model with the same compute budget?

2. **Diversity vs Quality**: Build two 100-example datasets: (A) 100 high-quality examples all from one category, (B) 100 medium-quality examples spread across 10 categories. Train models on both. Which generalises better?

3. **Deduplication Threshold Search**: Run semantic dedup with thresholds [0.80, 0.85, 0.90, 0.95, 0.99]. Plot (threshold, dataset size, eval loss after training). Find the sweet spot.

4. **Response Length Distribution**: Fine-tune models on datasets with different mean response lengths (50 words, 150 words, 400 words). Test whether models trained on verbose data produce verbose outputs even for simple questions.

5. **Self-Instruct Pipeline**: Implement the full Self-Instruct loop with an actual LLM API: generate 10 instructions from seed tasks, generate responses, filter, and add to the pool. Repeat 5 rounds. How does data diversity grow?

---

## 11. References

- [Wang et al. (2022) — Self-Instruct: Aligning Language Models with Self-Generated Instructions](https://arxiv.org/abs/2212.10560)
- [Chen et al. (2023) — Alpagasus: Training a Better Alpaca with Fewer Data](https://arxiv.org/abs/2307.08701)
- [Mukherjee et al. (2023) — Orca: Progressive Learning from Complex Explanation Traces of GPT-4](https://arxiv.org/abs/2306.02707)
- [Penedo et al. (2023) — The RefinedWeb Dataset for Falcon LLM: Outperforming Curated Corpora with Web Data](https://arxiv.org/abs/2306.01116)
- [Lee et al. (2021) — Deduplicating Training Data Makes Language Models Better](https://arxiv.org/abs/2107.06499)
- [HuggingFace Datasets Documentation](https://huggingface.co/docs/datasets)